# HADDOCK3 Install and Example

## Here are possible install routes

### Normal way

In [ ]:
!conda create -y -n haddock python=3.13

Activate haddocak

In [ ]:
!conda install -y -c conda-forge c-compiler cxx-compiler
%pip install --upgrade pip setuptools
#%pip install freesasa
%pip install haddock3

Or with MPI4py

In [ ]:
%pip install 'haddock3[mpi]'

### From the source

In [ ]:
!git clone https://github.com/haddocking/haddock3.git
!cd haddock3
!pip install .

## Running HADDOCK

## Automated HADDOCK3 Workflow
This cell handles PDB preprocessing, run directory archiving, and optimized docking parameters.

In [11]:
import os
import shutil
from datetime import datetime
from pathlib import Path
from haddock.clis.cli import main

# === CONFIGURATION ===
mol1_path = "../Data/TIMP3_Xray.pdb"
mol2_path = "../Data/MMP2_Xray.pdb"

# Output and Archive base
output_base = "../Local/haddock_data"
run_name = "haddock3_test_run"
run_dir = os.path.join(output_base, run_name)
past_runs_base = os.path.join(output_base, "past_runs")
# =====================

def modify_chain(in_file, out_file, new_chain):
    with open(in_file, 'r') as f_in, open(out_file, 'w') as f_out:
        for line in f_in:
            if line.startswith(('ATOM  ', 'HETATM', 'TER   ')):
                if len(line) >= 22:
                    f_out.write(line[:21] + new_chain + line[22:])
                else:
                    f_out.write(line)
            else:
                f_out.write(line)

os.makedirs(output_base, exist_ok=True)
os.makedirs(past_runs_base, exist_ok=True)

# 1. Archive previous run if exists
if os.path.exists(run_dir):
    now = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    archive_path = os.path.join(past_runs_base, now)
    os.makedirs(archive_path, exist_ok=True)
    print(f"Moving existing run to {archive_path}")
    shutil.move(run_dir, archive_path)

# 2. Prepare structures with distinct chains
haddock_data_tmp = os.path.join(output_base, "input_pdbs")
os.makedirs(haddock_data_tmp, exist_ok=True)
p1 = os.path.join(haddock_data_tmp, "mol1_fixed.pdb")
p2 = os.path.join(haddock_data_tmp, "mol2_fixed.pdb")
modify_chain(mol1_path, p1, "A")
modify_chain(mol2_path, p2, "B")

# 3. Create HADDOCK3 config
config_content = f"""
run_dir = \"{run_dir}\"
mode = \"local\"
ncores = 14
postprocess = true
clean = false

molecules = [
    \"{p1}\",
    \"{p2}\"
]

[topoaa]
[rigidbody]
sampling = 100
cmrest = true
[caprieval]
[flexref]
tolerance = 5
[emref]
[caprieval]
"""
config_file = os.path.join(output_base, "run.cfg")
with open(config_file, "w") as f:
    f.write(config_content)

print(f"Starting HADDOCK3 run: {run_dir}")
# 4. Run HADDOCK3
main(config_file)


Starting HADDOCK3 run: ../Local/haddock_data/haddock3_test_run
[2026-02-27 18:06:16,988 cli INFO] 
##############################################
#                                            #
#                 HADDOCK3                   #
#                                            #
##############################################

!! Some of the HADDOCK3 components use CNS (Crystallographic and NMR System) which is free of use for non-profit applications. !!
!! For commercial use it is your own responsibility to have a proper license. !!
!! For details refer to the DISCLAIMER file in the HADDOCK3 repository. !!

Starting HADDOCK3 v2025.11.0 on 2026-02-27 18:06:00

Python 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]

[2026-02-27 18:06:17,812 libworkflow INFO] Reading instructions step 0_topoaa
[2026-02-27 18:06:17,813 libworkflow INFO] Reading instructions step 1_rigidbody
[2026-02-27 18:06:17,813 libworkflow INFO] Reading instructions step 2_caprieval
[2026-02-27 18:06:17,814 l

## Visualize Best Complex
Identify and display the top-scoring docked model.

In [15]:
import py3Dmol
import pandas as pd
import os

# Configuration variables (should match the setup cell)
output_base = "../Local/haddock_data"
run_name = "haddock3_test_run"
run_dir = os.path.join(output_base, run_name)

if not os.path.exists(run_dir):
    print(f"Run directory not found: {run_dir}")
else:
    # Find the last caprieval step
    steps = sorted([d for d in os.listdir(run_dir) if "caprieval" in d])
    if steps:
        last_step = steps[-1]
        score_file = os.path.join(run_dir, last_step, "capri_ss.tsv")
        
        if os.path.exists(score_file):
            df = pd.read_csv(score_file, sep="\t")
            best_pdb_rel = df.sort_values("score").iloc[0]["model"]
            best_model_name = os.path.basename(best_pdb_rel)
            
            found_path = None
            for root, dirs, files in os.walk(run_dir):
                if best_model_name in files:
                    found_path = os.path.join(root, best_model_name)
                    break
            
            if found_path:
                print(f"Displaying best model: {found_path}")
                view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js')
                with open(found_path, "r") as f:
                    view.addModel(f.read(), "pdb")
                
                view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'spectrum'}})
                view.setStyle({'chain': 'B'}, {'cartoon': {'color': 'lightblue'}})
                view.zoomTo()
                view.show()
            else:
                print(f"Could not find model file: {best_model_name}")
        else:
            print(f"Score file not found: {score_file}")
    else:
        print("No caprieval steps found.")


Displaying best model: ../Local/haddock_data/haddock3_test_run/4_emref/emref_1.pdb


3Dmol.js failed to load for some reason. Please check your browser console for error messages.